# Part 1. Prompt Engineering

- A **prompt** is a **set of instructions** (including content   like text, images, etc) provided to an LLM to perform a task.

- **Prompt Engineering** is the process of crafting such the set of instructions that gets an LLM model to generate the desired outcome (i.e., solving the task). 

#### Components of Well-Structured Prompts
- **Role**: The role the LLM should adopt.
- **Task Description**: The specific instruction or question.
- **Context**: Additional information needed for the task.
- **Output Format**: How the response should be structured.
- **Examples (a.k.a. Few-Shot Learning)** (optional): Sample input/output pairs.

For the "Prompt Engineering and Structured Outputs" section of our tutorial, we will rely on OpenAI Software Development Kit (OpenAI SDK), which can be installed by running `pip install openai` (or similar).
We will use models running within Ollama and OpenAI SDK to connect to Ollama.

In [ ]:
# Import OpenAI client class
from openai import OpenAI

# Import other modules
import textwrap

In [ ]:
# Connect to Ollama running on the backend

openai_client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

In [ ]:
# Define a wrapper function that will send requests to an LLM and receive responses
def generate_response(
        client: OpenAI,
        model: str,
        user_prompt: str,
        system_prompt: str = "You are a helpful assistant.",
        temperature: float = 0.5       
) -> str:
    """Sends a request to an LLM and and returns a response."""
    
    response = client.chat.completions.create(
        model=model,
        temperature=temperature,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )

    return response.choices[0].message.content

#### 1. Vague vs. Precise Prompt
Let's see how an LLM's output changes when we start from a vague prompt, then adding a persona/role, and finally specifying the task.

In [ ]:
# Vague prompt = vague result.
# We should expect a generic output from the LLM.
vague_prompt = "Tell me about Albert Einstein."

vague_response = generate_response(
    client=openai_client,
    model="llama3.1",
    user_prompt=vague_prompt, 
)

print(vague_response)
# textwrap.fill(vague_response, width=80)

In [ ]:
# Add role (persona) to the vague prompt.
# The tone of the output should match the role provided in the prompt.
vague_system_prompt_with_persona = "You are a high-school physics teacher."
vague_user_prompt_with_persona = "Tell me about Albert Einstein."


response_with_persona = generate_response(
    client=openai_client,
    model="llama3.1",
    system_prompt=vague_system_prompt_with_persona,
    user_prompt=vague_user_prompt_with_persona,
)

print(response_with_persona)
# textwrap.fill(response_with_persona, width=80)

In [ ]:
# Finally let's narrow down the scope of the output by providing 
# specific task and the output format.
precise_system_prompt = "You are a high-school physics teacher."

precise_user_prompt = """
Explain the role Albert Einstein played in the development of physics 
in 20th century.
Write two paragraphs.
"""

precise_response = generate_response(
    client=openai_client,
    model="llama3.1",
    system_prompt=precise_system_prompt,
    user_prompt=precise_user_prompt
)

print(precise_response)
# textwrap.fill(precise_response, width=80)

In the case of the vague prompt, we can see that the output text covers various aspects of Albert Einstein's live without providing depth into any of those. Advancing forward, when we provide a role ("a high-school physics teacher"), the output changes its format and is "tailored" to the role/persona specified. 
Finally, with the precise prompt, we **guide** an LLM to produce a specific but more detailed response. While for some cases (e.g., famous personas) we might want to learn both perspectives (in breadth and depth), for AI applications we typically want to narrow down the scope of the task to get more specific outputs.

So, **Best Practice #1: Be specific and direct.**

#### 2. Adding Context
Any LLM model "knows" only what it was trained on.
However, you can incorporate new information by providing 
it as a context.

In [ ]:
# Source: https://tacc.utexas.edu/about/staff-directory/niall-gaffney/
context = """
NIALL GAFFNEY

Data and AI Directorate

Phone: 512-475-9504 | Email: ngaffney@tacc.utexas.edu

Education: B.A., M.A., Ph.D., Astronomy University of Texas at Austin

Niall Gaffney's background primarily revolves around the management and utilization of 
large inhomogeneous scientific datasets. Niall, who earned his B.A., M.A., and Ph.D. 
degrees in astronomy from The University of Texas at Austin, joined TACC in May 2013. 
Most of his focus has been on creating environments to foster better data practices 
from improving metadata, data processing, analysis, and reuse. He focuses on improving 
researchers' data practices to accelerate outcomes and better feed the Machine Learning 
and Artificial Intelligence applications which are becoming more broadly adopted in 
science and engineering research fields. Much of this stems from his 13 years as designer 
and developer for the archives at the Space Telescope Science Institute (STScI), which 
holds the data from the Hubble Space Telescope, Kepler, and James Webb Space Telescope 
missions.

He was also a leader in developing the Hubble Legacy Archive. This project harvested 
the 20+ years of Hubble Space Telescope data to create some of the most sensitive 
astronomical data products available for open research. Before his work at STScI, 
Niall was worked as "the friend of the telescope" for the Hobby Eberly Telescope (HET) 
project at the McDonald Observatory in west Texas. This was the start of his work in 
planning experiments and then cataloging the data the HET produced.
"""

**Temperature** is the parameter for controlling the randomness (or creativity) of a model. By increasing the temperature from 0 to 1, we're increasing the randomness (hence more creativity) of the response. 

In [ ]:
# First let's see what a model "knows" about Niall Gaffney.
# In other words, do NOT provide any context first. 

# Tip: Modify temperature and see how this parameter influences
# the output of the model by sending requests multiple times.
no_context_prompt = "Who is Niall Gaffney?"

response_wo_context = generate_response(
    client=openai_client,
    model="llama3.1",
    user_prompt=no_context_prompt,
    temperature=1.0
)

print(response_wo_context)

In [ ]:
# Now, let's add the context.
# Tip: run this part multiple times for temperature=0.0 and temperature=1.0
prompt_with_context = no_context_prompt + f"\nContext: {context}"

response_with_context = generate_response(
    client=openai_client,
    model="llama3.1",
    user_prompt=prompt_with_context,
    temperature=0.0
)

print(response_with_context)

We've seen that by providing context, we can "add" new information/knowledge to a model without retraining it. What's more, by providing context we can also reduce the occurrence of hallucinations [1]. However, note that adding the context **does NOT guarantee** that a model will strictly follow it [1].

Also, adding additional instructions like "generate response based on the context provided" to the prompt can also be helpful.

**Best Practice #2: Add specific context to your prompts when applicable.**

[1] Huyen, C. (2024). Prompt Engineering. In AI Engineering: Building Applications with Foundation Models. (pp. 211-252) O'Reilly Media, Inc.

#### 3. Prompt Chaining
Break complex tasks into simpler subtasks.
Use each LLM's output as a **context** for the next prompt/step.

In [ ]:

topic = "the impact of Albert Einstein on philosophy of 20th century"

# Step 1: Create an outline for the blog post
prompt_step_1 = f"Come up with a 3 point outline for a post about: {topic}"

response_step_1 = generate_response(
    client=openai_client,
    model="llama3.1",
    user_prompt=prompt_step_1,
    temperature=0.7
)

print(response_step_1)

In [ ]:
# Step 2: Write introduction using the outline
prompt_step_2=f"""
Using the following OUTLINE, write an introduction paragraph with 80-100 words.
OUTLINE: {response_step_1}
Hook the reader with a surprising fact in the first sentence.
"""

response_step_2 = generate_response(
    client=openai_client,
    model="llama3.1",
    user_prompt=prompt_step_2,
    temperature=0.7
)

print(response_step_2)

In [ ]:
# Step 3: Come up with a few options for the title
prompt_step_3 = f"""
Based on the INTRODUCTION below, come up with 3 catchy blog post titles.
INTRODUCTION: {response_step_2}
Format your output as 3 bullet points.
""" 
response_step_3 = generate_response(
    client=openai_client,
    model="llama3.1",
    user_prompt=prompt_step_3,
    temperature=1
)

print(response_step_3)

Main motivation behind the prompt chaining technique: a few smaller prompts are better than one that is a giant one. If your application require solving complex tasks with multiple steps, divide them into smaller subtasks by introducing smaller prompts and chain LLM's outputs together with the following prompts.

Remember: When working with LLMs, simpler instructions are better than complex ones.

**Best Practice #3: Break complex prompts into smaller ones.**

#### 4. More Practice - Code Generation

In [ ]:
role = "a Python Developer"
task_description = "to develop Python code based on the specification provided"
context = "write a Python function that generates Fibonacci sequence; use a `while` loop"
output_format = """\n
    **EXPLANATION**: [provide a brief explanation of your solution]
    **PYTHON CODE**: [a block of Python code]
"""

# Do not provide additional text after the **PYTHON CODE** section

In [ ]:
system_prompt = f"You are {role}. Your task is {task_description}."
user_prompt = f"""
Here are additional instructions: {context}.
OUTPUT FORMAT: {output_format}
"""

print("SYSTEM PROMPT:", system_prompt, sep="\n")
print("-" * 100)
print("USER PROMPT:", user_prompt, sep="\n")

In [ ]:
response = generate_response(
    client=openai_client,
    model="llama3.1",
    user_prompt=user_prompt,
    system_prompt=system_prompt,
)

print(response)